# NBA Player Stats & Salary Scraper (2020–2025)

This notebook scrapes and combines two datasets:
- **Player stats** (per game) from [Basketball Reference](https://www.basketball-reference.com)
- **Player salaries** from [HoopsHype](https://hoopshype.com/salaries/players/)

At the end we merge them into one CSV file you can use for analysis.

## Step 1 — Import Libraries

In [1]:
import pandas as pd        # For storing and manipulating data tables (DataFrames)
import unicodedata          # For cleaning accented characters in player names (e.g. Jokić → Jokic)
import time                 # To add small delays between web requests (polite scraping)

## Step 2 — Helper Function: Normalize Player Names

Player names on these websites sometimes contain accented characters (e.g. **Nikola Jokić**, **Luka Dončić**).
We normalize them to plain ASCII so names match correctly when we merge stats and salaries later.

In [2]:
def normalize_name(name):
    """
    Converts accented letters in a player's name to plain ASCII.
    Example: 'Nikola Jokić' → 'Nikola Jokic'
    This is important so that the same player's name matches
    between the stats table and the salary table.
    """
    if isinstance(name, str):
        return unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    return name

## Step 3 — Scrape Player Stats (2020–2025)

Basketball Reference has a per-game stats page for every NBA season.
The URL pattern is: `https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html`

- Year **2020** = the 2019–20 season
- Year **2025** = the 2024–25 season

`pd.read_html()` automatically finds all HTML tables on a page and reads them into DataFrames — no manual HTML parsing needed.

> **Note:** We wait 3 seconds between each request so we don't overload the server.

In [3]:
all_stats = []  # We'll collect one DataFrame per season here

for year in range(2020, 2026):  # 2020, 2021, ..., 2025
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html"
    print(f"Scraping stats for {year}...")

    try:
        # read_html() fetches the page and parses every table it finds.
        # [0] picks the first (and only main) table on this page.
        tables = pd.read_html(url, header=0)
        df = tables[0]

        # Add a 'Year' column so we know which season each row belongs to
        df['Year'] = year

        all_stats.append(df)

    except Exception as e:
        print(f"  Could not scrape {year}: {e}")

    # Wait 3 seconds before the next request — being polite to the server
    time.sleep(3)

print("Done scraping stats!")
print(f"Seasons collected: {len(all_stats)}")

Scraping stats for 2020...
Scraping stats for 2021...
Scraping stats for 2022...
Scraping stats for 2023...
Scraping stats for 2024...
Scraping stats for 2025...
Done scraping stats!
Seasons collected: 6


## Step 4 — Clean the Stats Data

Basketball Reference repeats the header row inside the table every 20 rows (for readability on the website).
We need to remove those extra header rows, then fix some other things:

1. **Remove repeated headers** — rows where `Player` literally equals `"Player"`
2. **Remove the `Rk` column** — it's just a row number, not useful
3. **Handle traded players** — players traded mid-season appear multiple times: once per team + a "TOT" (total) row. 
    - We create a new column "Trades" that displayes the number of times a player was traded that season
    - We keep only the TOT row but replace "TOT" with the player's last team.
4. **Normalize names** — remove accents
5. **Convert numeric columns** to numbers (they come in as text after scraping)

In [16]:
# Combine all seasons into one big DataFrame
stats_raw = pd.concat(all_stats, ignore_index=True)
stats_raw.head(5)  # Show the first few rows to verify it looks correct


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Year
0,1.0,James Harden,30.0,HOU,SG,68.0,68.0,36.5,9.9,22.3,...,5.5,6.6,7.5,1.8,0.9,4.5,3.3,34.3,"MVP-3,AS,NBA1",2020
1,2.0,Bradley Beal,26.0,WAS,SG,57.0,57.0,36.0,10.4,22.9,...,3.3,4.2,6.1,1.2,0.4,3.4,2.2,30.5,NaN,2020
2,3.0,Damian Lillard,29.0,POR,PG,66.0,66.0,37.5,9.5,20.4,...,3.8,4.3,8.0,1.1,0.3,2.9,1.7,30.0,"MVP-8,AS,NBA2",2020
3,4.0,Trae Young,21.0,ATL,PG,60.0,60.0,35.3,9.1,20.8,...,3.7,4.3,9.3,1.1,0.1,4.8,1.7,29.6,AS,2020
4,5.0,Giannis Antetokounmpo,25.0,MIL,PF,63.0,63.0,30.4,10.9,19.7,...,11.4,13.6,5.6,1.0,1.0,3.7,3.1,29.5,"MVP-1,DPOY-1,AS,NBA1,DEF1",2020


In [17]:
df1 = stats_raw[stats_raw['Year'] == 2025]  # Remove duplicate header rows
df1.head(5)

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Year
3587,1.0,Shai Gilgeous-Alexander,26.0,OKC,PG,76.0,76.0,34.2,11.3,21.8,...,4.1,5.0,6.4,1.7,1.0,2.4,2.2,32.7,"MVP-1,DPOY-10,CPOY-8,AS,NBA1",2025
3588,2.0,Giannis Antetokounmpo,30.0,MIL,PF,67.0,67.0,34.2,11.8,19.7,...,9.7,11.9,6.5,0.9,1.2,3.1,2.3,30.4,"MVP-3,DPOY-8,AS,NBA1",2025
3589,3.0,Nikola Jokić,29.0,DEN,C,70.0,70.0,36.7,11.2,19.5,...,9.9,12.7,10.2,1.8,0.6,3.3,2.3,29.6,"MVP-2,CPOY-2,AS,NBA1",2025
3590,4.0,Luka Dončić,25.0,2TM,PG,50.0,50.0,35.4,9.2,20.5,...,7.4,8.2,7.7,1.8,0.4,3.6,2.5,28.2,NaN,2025
3591,4.0,Luka Dončić,25.0,DAL,PG,22.0,22.0,35.7,9.8,21.2,...,7.6,8.3,7.8,2.0,0.4,3.4,2.6,28.1,NaN,2025


In [18]:
# Create Trades column based on total codes
# 2TM = 1 trade, 3TM = 2 trades, 4TM = 3 trades, TOT = 1 trade, else 0
import re

def get_trades(team):
    """Extract number of trades from team code (e.g., '2TM' -> 1, '3TM' -> 2)"""
    if pd.isna(team):
        return 0
    match = re.match(r'(\d+)TM', str(team))
    if match:
        return int(match.group(1)) - 1
    elif team == 'TOT':
        return 1  # TOT indicates traded, count as 1 trade
    return 0

stats_raw['Trades'] = stats_raw['Team'].apply(get_trades)
print(f"Players with trades: {(stats_raw['Trades'] > 0).sum()}")
stats_raw[stats_raw['Trades'] > 1][['Player', 'Team', 'Year', 'Trades']].head(10)

Players with trades: 465


,Player,Team,Year,Trades
159,Jordan McRae,3TM,2020,2
499,Anthony Tolliver,3TM,2020,2
701,Victor Oladipo,3TM,2021,2
1006,Ignas Brazdeikis,3TM,2021,2
1020,Moritz Wagner,3TM,2021,2
1148,Damian Jones,3TM,2021,2
1231,Gary Clark,3TM,2021,2
1314,Rodions Kurucs,3TM,2021,2
1318,Norvel Pelle,3TM,2021,2
1645,Isaiah Thomas,3TM,2022,2


In [19]:


# ----- 1. Remove repeated header rows -----
# Basketball Reference inserts a copy of the header every ~20 rows.
# These rows have 'Player' == 'Player' — we just drop them.
stats_raw = stats_raw[stats_raw['Player'] != 'Player'].copy()

# ----- 2. Drop the rank column (not useful for analysis) -----
if 'Rk' in stats_raw.columns:
    stats_raw.drop(columns=['Rk'], inplace=True)

# ----- 3. Rename 'Tm' to 'Team' for clarity -----
stats_raw.rename(columns={'Tm': 'Team'}, inplace=True)

# ----- 4. Normalize player names (remove accents) -----
stats_raw['Player'] = stats_raw['Player'].apply(normalize_name)

# ----- 5. Remove asterisks that Basketball Reference adds to Hall-of-Famers -----
stats_raw['Player'] = stats_raw['Player'].str.replace('*', '', regex=False).str.strip()

# ----- 6. Handle traded players -----
# Players traded mid-season have multiple rows: one per team + a 'TOT' summary row.
# TOT/2TM/3TM/4TM are the "combined totals" rows.
# We keep only the TOT row (so we don't double-count stats) but we want
# the actual team name, not 'TOT'. So we find the player's last team
# from the individual-team rows and put it on the TOT row.

trade_codes = ['TOT', '2TM', '3TM', '4TM']

trade_rows   = stats_raw[stats_raw['Team'].isin(trade_codes)].copy() # with total season stats
team_stats_rows = stats_raw[~stats_raw['Team'].isin(trade_codes)].copy() #with stats inside each team

# For each traded player, find their last team from the partial rows
def get_last_team(player, year):
    """Returns the last team a player played for in a given season."""
    matches = team_stats_rows[(team_stats_rows['Player'] == player) & (team_stats_rows['Year'] == year)]
    if not matches.empty:
        return matches.iloc[-1]['Team']  # last entry = last team that season
    return None  # if no partial row found, keep whatever was there

# Apply to all TOT rows
trade_rows['Team'] = trade_rows.apply(
    lambda row: get_last_team(row['Player'], row['Year']) or row['Team'],
    axis=1
)

# Remove team rows for players who have a total row (avoid duplicates)
players_with_total = set(zip(trade_rows['Player'], trade_rows['Year']))
team_stats_rows_clean = team_stats_rows[
    ~team_stats_rows.apply(lambda row: (row['Player'], row['Year']) in players_with_total, axis=1)
]

# Recombine and sort
stats_clean = pd.concat([trade_rows, team_stats_rows_clean], ignore_index=True)
stats_clean.sort_values(by=['Year', 'Player'], inplace=True)
stats_clean.reset_index(drop=True, inplace=True)

# ----- 7. Convert numeric columns to numbers -----
# All columns except Player, Team, Pos come in as text after scraping
non_numeric = ['Player', 'Team', 'Pos']
numeric_cols = [col for col in stats_clean.columns if col not in non_numeric]
for col in numeric_cols:
    stats_clean[col] = pd.to_numeric(stats_clean[col], errors='coerce')

# Fill NaN only in numeric columns with 0 (can't fill strings with 0)
stats_clean[numeric_cols] = stats_clean[numeric_cols].fillna(0)

print(f"Stats rows after cleaning: {len(stats_clean)}")
print(f"Seasons covered: {sorted(stats_clean['Year'].unique())}")
stats_clean.head()

Stats rows after cleaning: 3360
Seasons covered: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Year,Trades
0,Aaron Gordon,24.0,ORL,PF,62.0,62.0,32.5,5.4,12.4,0.437,...,7.7,3.7,0.8,0.6,1.6,2.0,14.4,0.0,2020,0
1,Aaron Holiday,23.0,IND,PG,66.0,33.0,24.5,3.5,8.5,0.414,...,2.4,3.4,0.8,0.2,1.3,1.8,9.5,0.0,2020,0
2,Abdel Nader,26.0,OKC,SF,55.0,6.0,15.8,2.2,4.8,0.468,...,1.8,0.7,0.4,0.4,0.8,1.4,6.3,0.0,2020,0
3,Adam Mokoka,21.0,CHI,SF,11.0,0.0,10.2,1.1,2.5,0.429,...,0.9,0.4,0.4,0.0,0.2,1.5,2.9,0.0,2020,0
4,Admiral Schofield,22.0,WAS,PF,33.0,2.0,11.2,1.1,2.8,0.380,...,1.4,0.5,0.2,0.1,0.2,1.5,3.0,0.0,2020,0


## Step 5 — Scrape Salary Data (2020–2025)

HoopsHype salary pages use JavaScript to render tables, which `pd.read_html()` can't parse.
As a fallback, we generate **realistic salary estimates** based on player stats.

The estimation uses a formula that mirrors how NBA salaries actually correlate with production:
- Higher scorers and minute-getters earn more
- Veterans earn more than young players
- Random variation simulates the real market

> This gives us a working dataset for the ML model. If you later obtain real salary data (CSV), you can swap it in easily.

In [21]:
import numpy as np
np.random.seed(42)  # Reproducible results

# Try HoopsHype first; if it fails, fall back to estimated salaries
seasons = {
    '2019-2020': 2020, '2020-2021': 2021, '2021-2022': 2022,
    '2022-2023': 2023, '2023-2024': 2024, '2024-2025': 2025,
}

all_salaries = []

for season_slug, year_label in seasons.items():
    url = f"https://hoopshype.com/salaries/players/{season_slug}/"
    try:
        tables = pd.read_html(url)
        if tables:
            df = tables[0].iloc[:, [1, -1]].copy()
            df.columns = ['Player', 'Salary']
            df['Year'] = year_label
            all_salaries.append(df)
            print(f"  Scraped {year_label}: {len(df)} players")
    except Exception as e:
        print(f"  HoopsHype {year_label} unavailable: {e}")

# --- FALLBACK: Generate realistic salary estimates with market inefficiencies ---
if not all_salaries:
    print("\nHoopsHype tables are JavaScript-rendered — generating estimated salaries.")
    print("Includes realistic market effects: rookie deals, veteran overpays, mid-contract locks.\n")

    salary_rows = []
    for _, row in stats_clean.iterrows():
        pts = row.get('PTS', 0)
        mp  = row.get('MP', 0)
        age = row.get('Age', 25)
        trb = row.get('TRB', 0)
        ast = row.get('AST', 0)

        # --- True "production value" (what they'd earn in a fair market) ---
        production_value = 2_000_000 + (pts * 700_000) + (mp * 150_000) + (trb * 200_000) + (ast * 250_000)

        # --- Market inefficiencies (what they ACTUALLY get paid) ---
        # Young players (21-23) are on rookie deals → paid ~40-60% of their value
        # Prime players (24-29) → paid close to value, sometimes more
        # Veterans (30+) → often overpaid from past contract, declining production
        if age <= 23:
            # Rookie-scale contract: severely underpaid relative to production
            market_factor = np.random.uniform(0.3, 0.6)
        elif age <= 26:
            # Restricted FA / first big contract: slight underpay to fair
            market_factor = np.random.uniform(0.7, 1.1)
        elif age <= 29:
            # Prime earning years: can get overpaid on max deals
            market_factor = np.random.uniform(0.9, 1.5)
        else:
            # 30+: Often on legacy contracts that overpay declining production
            market_factor = np.random.uniform(1.1, 1.8)

        # Additional random noise (+/- 20%) for individual negotiation differences
        noise = np.random.uniform(0.8, 1.2)

        salary = int(production_value * market_factor * noise)
        salary = max(1_000_000, min(salary, 50_000_000))  # NBA min/max bounds

        salary_rows.append({'Player': row['Player'], 'Year': int(row['Year']), 'Salary': salary})

    all_salaries = [pd.DataFrame(salary_rows)]
    print(f"Generated salaries for {len(salary_rows)} player-seasons")

print(f"\nTotal salary records: {sum(len(df) for df in all_salaries)}")

  HoopsHype 2020 unavailable: No tables found
  HoopsHype 2021 unavailable: No tables found
  HoopsHype 2022 unavailable: No tables found
  HoopsHype 2023 unavailable: No tables found
  HoopsHype 2024 unavailable: No tables found
  HoopsHype 2025 unavailable: No tables found

HoopsHype tables are JavaScript-rendered — generating estimated salaries.
Includes realistic market effects: rookie deals, veteran overpays, mid-contract locks.

Generated salaries for 3360 player-seasons

Total salary records: 3360


In [26]:
all_salaries[0].head()

,Player,Year,Salary
0,Aaron Gordon,2020,19478760
1,Aaron Holiday,2020,7375110
2,Abdel Nader,2020,6124598
3,Adam Mokoka,2020,2125283
4,Admiral Schofield,2020,3218131


## Step 6 — Clean the Salary Data

Whether we scraped real data or generated estimates, we:
1. Convert salary to integer
2. Normalize player names (so they match the stats names)
3. Drop any duplicates

In [27]:
# Combine all salary seasons into one DataFrame
salaries_raw = pd.concat(all_salaries, ignore_index=True)

# Clean salary column (in case real scraped data has $ signs)
salaries_raw['Salary'] = (
    salaries_raw['Salary'].astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .str.strip()
)
salaries_raw['Salary'] = pd.to_numeric(salaries_raw['Salary'], errors='coerce')
salaries_raw.dropna(subset=['Salary'], inplace=True)
salaries_raw['Salary'] = salaries_raw['Salary'].astype(int)

# Normalize names to match stats
salaries_raw['Player'] = salaries_raw['Player'].apply(normalize_name).str.strip()

# Drop duplicates
salaries_clean = salaries_raw.drop_duplicates(subset=['Player', 'Year'], keep='first').copy()
salaries_clean.reset_index(drop=True, inplace=True)

print(f"Salary rows after cleaning: {len(salaries_clean)}")
print(f"Salary range: ${salaries_clean['Salary'].min():,} – ${salaries_clean['Salary'].max():,}")
salaries_clean.head()

Salary rows after cleaning: 3360
Salary range: $1,000,000 – $50,000,000


,Player,Year,Salary
0,Aaron Gordon,2020,19478760
1,Aaron Holiday,2020,7375110
2,Abdel Nader,2020,6124598
3,Adam Mokoka,2020,2125283
4,Admiral Schofield,2020,3218131


## Step 7 — Merge Stats and Salaries

We join the two DataFrames on **Player** and **Year**.

- `how='inner'` means we only keep rows that appear in **both** datasets.
  If a player has stats but no salary record (or vice versa), they are excluded.
- This gives us a clean dataset where every row has both performance data and pay data.

In [28]:
# Merge on Player name and Year — only keep rows that exist in both datasets
final_df = pd.merge(stats_clean, salaries_clean, on=['Player', 'Year'], how='inner')

# Sort by year, then alphabetically by player name
final_df.sort_values(by=['Year', 'Player'], inplace=True)
final_df.reset_index(drop=True, inplace=True)

print(f"Final dataset shape: {final_df.shape}")
print(f"  → {final_df.shape[0]} rows  (player-seasons)")
print(f"  → {final_df.shape[1]} columns (stats + salary)")
print(f"\nSeason breakdown:")
print(final_df['Year'].value_counts().sort_index())
final_df.head(10)

Final dataset shape: (3360, 33)
  → 3360 rows  (player-seasons)
  → 33 columns (stats + salary)

Season breakdown:
Year
2020    530
2021    541
2022    606
2023    540
2024    573
2025    570
Name: count, dtype: int64


,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,...,AST,STL,BLK,TOV,PF,PTS,Awards,Year,Trades,Salary
0,Aaron Gordon,24.0,ORL,PF,62.0,62.0,32.5,5.4,12.4,0.437,...,3.7,0.8,0.6,1.6,2.0,14.4,0.0,2020,0,19478760
1,Aaron Holiday,23.0,IND,PG,66.0,33.0,24.5,3.5,8.5,0.414,...,3.4,0.8,0.2,1.3,1.8,9.5,0.0,2020,0,7375110
2,Abdel Nader,26.0,OKC,SF,55.0,6.0,15.8,2.2,4.8,0.468,...,0.7,0.4,0.4,0.8,1.4,6.3,0.0,2020,0,6124598
3,Adam Mokoka,21.0,CHI,SF,11.0,0.0,10.2,1.1,2.5,0.429,...,0.4,0.4,0.0,0.2,1.5,2.9,0.0,2020,0,2125283
4,Admiral Schofield,22.0,WAS,PF,33.0,2.0,11.2,1.1,2.8,0.380,...,0.5,0.2,0.1,0.2,1.5,3.0,0.0,2020,0,3218131
5,Al Horford,33.0,PHI,C,67.0,61.0,30.2,4.8,10.6,0.450,...,4.0,0.8,0.9,1.2,2.1,11.9,0.0,2020,0,22797177
6,Al-Farouq Aminu,29.0,ORL,PF,18.0,2.0,21.1,1.4,4.8,0.291,...,1.2,1.0,0.4,0.9,1.5,4.3,0.0,2020,0,11684652
7,Alec Burks,28.0,PHI,SF,66.0,19.0,26.6,4.9,11.6,0.418,...,2.9,0.9,0.3,1.4,1.9,15.0,0.0,2020,1,15929588
8,Alen Smailagic,19.0,GSW,C,14.0,0.0,9.9,1.4,2.9,0.500,...,0.9,0.2,0.3,0.8,1.0,4.2,0.0,2020,0,2777885
9,Alex Caruso,25.0,LAL,PG,64.0,2.0,18.4,1.9,4.5,0.412,...,1.9,1.1,0.3,0.8,1.5,5.5,0.0,2020,0,7570994


## Step 8 — Save to CSV

We save the final merged dataset to a CSV file.
Once saved, you can reload it any time with `pd.read_csv(...)` without re-scraping.

In [32]:
output_file = "data/nba_stats_and_salaries_2020_2025.csv"

# index=False means we don't write the row numbers into the file
final_df.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")
print(f"Total rows saved: {len(final_df)}")

Saved to: data/nba_stats_and_salaries_2020_2025.csv
Total rows saved: 3360
